# Imports

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
import random
import torch
import re
 
print("Libraries loaded ")

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Cuda check and Wandb Setup

**Cuda setup**

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

**WANDB Setup**

In [ ]:
import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("WANDB_API_KEY")
wandb.login(key=wandb_key)

PROJECT_NAME = "DL-GenAi-2026-T2" 
print("W&B login successful")

# Map@3 Metric function

In [ ]:
def map_at_3(y_true, y_pred_ranked):
    scores = []
    for true, preds in zip(y_true, y_pred_ranked):
        score = 0.0
        for i, p in enumerate(preds[:3]):
            if p == true:
                score = 1.0 / (i + 1)
                break
        scores.append(score)
    return np.mean(scores)

# Loading datasets

In [ ]:
BASE = '/kaggle/input/competitions/smart-mcq-solver-challenge'
 
train = pd.read_csv(f'{BASE}/train.csv')
test = pd.read_csv(f'{BASE}/test.csv')
sub = pd.read_csv(f'{BASE}/sample_submission.csv')
 
OPTIONS = list('ABCDE')
 
print(f"Train : {train.shape}")
print(f"Test : {test.shape}")
print(f"Sub : {sub.shape}")

# EDA

In [ ]:
# Datatypes of collumns of all datasets
print("=== TRAIN COLUMNS ===")
print(train.dtypes)
print("\n=== TEST COLUMNS ===")
print(test.dtypes)
print("\n=== SAMPLE SUBMISSION ===")
print(sub.head())

Since all columns are object type(text) and there is no numerical data this is purely a text/NLP problem.

In [ ]:
# First look of data
print("=== TRAIN HEAD ===")
display(train.head(3))
print("\n=== TEST HEAD ===")
display(test.head(3))

As we can see the options are tricky and very similar.The main challenge is observing differences between options and not just knowing the topic broadly due to the similarity between options.

In [ ]:
# Specific Row look
row = train.iloc[7]
print("PROMPT:", row['prompt'])
for opt in OPTIONS:
    print(f"[{opt}]", row[opt])
print("\nCORRECT ANSWER:", row['answer'])

Just as we had seen in the previous first look of data here we can see that the model needs to know the precise definitions and not just similar type of concepts beacuse the dataset is delibirately putting related but wrong concepts as distractors.

In [ ]:
#Missing values
print("=== TRAIN — NULL VALUES ===")
print(train.isnull().sum())
print("\n=== TEST — NULL VALUES ===")
print(test.isnull().sum())
# Duplicate check
print(f"Duplicate prompts in train : {train['prompt'].duplicated().sum()}")
print(f"Duplicate prompts in test  : {test['prompt'].duplicated().sum()}")

There are absolutely no missing data anywhere so no need to do null handling or imputation but the duplicate prompts in train will confuse and damage our models performance if same prompt gives different answers so lets check that.

In [ ]:
# Checking if duplicate prompts have same or different answers
dup = train[train.duplicated(subset='prompt', keep=False)]
print(dup[['prompt', 'answer']].sort_values('prompt'))

conflict = dup.groupby('prompt')['answer'].nunique()
print("\nPrompts with conflicting answers:", (conflict > 1).sum())

Since we have same duplicates it wont have that much effect on model training but still the model will give more importance to the duplicated prompts creating bias towards repesated questions so we will just drop them.

In [ ]:
train = train.drop_duplicates(subset='prompt', keep='first')
print("Train size after removing duplicates:", len(train))

In [ ]:
# Answer Distribution visualization
ans_counts = train['answer'].value_counts().sort_index()
print("=== ANSWER DISTRIBUTION ===")
print(ans_counts)

# Barchart demonstration for distributiuon
ans_counts.plot(kind='bar', color='steelblue', figsize=(6,4))
plt.title('Answer Distribution')
plt.xlabel('Option')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

B is most common (430) and E is least common (287) hence data is slightly imbalanced but our models won't be heavily biased toward predicting one letter over others as none of the option is dominant or rare.

In [ ]:
# Text length visualization
print("=== PROMPT LENGTH ===")
print(train['prompt'].str.len().describe().round(1))
print("\n=== AVERAGE OPTION LENGTH ===")
for opt in OPTIONS:
    avg = train[opt].str.len().mean()
    print(f"  Option {opt}: {avg:.1f} chars")

# Box plot of option lengths
length_data = [train[opt].str.len().values for opt in OPTIONS]
plt.boxplot(length_data, labels=OPTIONS)
plt.title('Option Text Length Distribution')
plt.xlabel('Option')
plt.ylabel('Character Count')
plt.tight_layout()
plt.show()

Similar character counts across all options so no conclusion with this.

In [ ]:
print("=== AVERAGE WORD COUNT PER OPTION ===")
for opt in OPTIONS:
    avg = train[opt].str.split().str.len().mean()
    print(f" Option {opt}: {avg:.1f} words")

word_data = [train[opt].str.split().str.len().values for opt in OPTIONS]
plt.boxplot(word_data, labels=OPTIONS)
plt.title('Word Count per Option')
plt.xlabel('Option')
plt.ylabel('Word Count')
plt.tight_layout()
plt.show()

All options average is almost identical except few outliers like B has one option near to 118 words and E has one option around 105 words but that wont have any effect.

In [ ]:
# Word overlap in prompt and correct answer
import re
def word_overlap(prompt, answer_text):
    prompt_words  = set(re.findall(r'\w+', prompt.lower()))
    answer_words  = set(re.findall(r'\w+', answer_text.lower()))
    common = prompt_words & answer_words
    return len(common)

train['correct_option_text'] = train.apply(lambda r: r[r['answer']], axis=1)
train['overlap_correct'] = train.apply(lambda r: word_overlap(r['prompt'], r['correct_option_text']), axis=1)

print("=== WORD OVERLAP(prompt → correct answer) ===")
print(train['overlap_correct'].describe().round(2))

train['overlap_correct'].plot(kind='hist', bins=30, figsize=(7,4))
plt.title('Word Overlap: Prompt vs Correct Answer')
plt.xlabel('Number of Common Words')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

Average overlap is 5 words, so correct answers typically share 5 words with the question  and aroound 205 questions have 0 word overlap hence Word overlap is too low and unreliable to be useful as a feature.

In [ ]:
# Correct vs wrong option's word count comparision
correct_lengths = []
wrong_lengths = []

for _, row in train.iterrows():
    for opt in OPTIONS:
        length = len(row[opt].split())
        if opt == row['answer']:
            correct_lengths.append(length)
        else:
            wrong_lengths.append(length)

print(f"Correct option avg word count : {np.mean(correct_lengths):.1f}")
print(f"Wrong option avg word count: {np.mean(wrong_lengths):.1f}")

plt.boxplot([correct_lengths, wrong_lengths], labels=['Correct', 'Wrong'])
plt.title('Correct vs Wrong Option Word Count')
plt.xlabel('Option Type')
plt.ylabel('Word Count')
plt.tight_layout()
plt.show()

Small but real pattern, correct answers are slightly longer with around ~3 words difference hence there is a tiny length bias in dataset.

In [ ]:
patterns = {
    'What is / What are' : r'^what (is|are)',
    'Which': r'^which',
    'Pick the best': r'^pick',
    'Select': r'^select',
    'Determine': r'^determine',
    'How': r'^how',
    'Why': r'^why',
    'Who': r'^who',
    'When': r'^when',
    'Other': None
}

def get_question_type(prompt):
    p = prompt.lower().strip()
    for label, pattern in patterns.items():
        if pattern and re.match(pattern, p):
            return label
    return 'Other'

train['question_type'] = train['prompt'].apply(get_question_type)
print("=== QUESTION TYPE COUNTS ===")
print(train['question_type'].value_counts())

train['question_type'].value_counts().plot(kind='bar', figsize=(8,4))
plt.title('Question Types in Train Set')
plt.xlabel('Type')
plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

Dataset is dominated by Other (513) and choice-based questions (Pick the best, Select, Which, Determine — ~257-277 each), with definition-based What is/What are (169) also present, while Who and How are nearly absent (7 and 2) — meaning the model must handle varied question formats with no single type to over-optimize for.

# Train Val Split

In [ ]:
train_df, val_df = train_test_split(
    train,
    test_size=0.15,
    random_state=SEED,
    stratify=train['answer']
)

# Sample Submission

In [ ]:
sub.to_csv('submission.csv', index=False)